# 📄 生成式 AI 應用開發｜第 9 週
## 文件處理與資料前處理（教師版）

本週把第 8 週只能處理文字貼上的 App，升級成能讀取 **PDF、Word、CSV、TXT 與 Markdown** 的文件工具。
重點不是立刻做完整 RAG，而是先建立可靠的資料管線：

**上傳檔案 → 判斷格式 → 擷取文字 → 清理 → 分塊（chunking）→ 預覽／摘要**

<font color="#C62828"><b>安全提醒：</b></font>
不要上傳含個資、機密或未授權內容的真實文件；API key 只能放在環境變數、`.env` 或 Streamlit Secrets。


## 0. 本週學習目標與三小時流程

完成本週後，你應能：

1. 依副檔名選擇 PDF、DOCX、CSV 或純文字的讀取方式。
2. 將不同格式轉成統一的 Python 字串。
3. 清理多餘空白，同時保留段落邊界。
4. 使用固定長度與 overlap 建立 chunks。
5. 說明 chunk 太大、太小與 overlap 過多的代價。
6. 在 Streamlit App 中預覽文件與 chunks，並選擇是否呼叫 AI 摘要。

| 時間 | 活動 |
|---|---|
| 50 分鐘 | 文件格式、抽取流程、清理與風險 |
| 80 分鐘 | Python 實作、chunking 與本機測試 |
| 30 分鐘 | Streamlit 專案、除錯與展示 |
| 20 分鐘 | 練習檢核與第 10 週 Embedding 預告 |


## 1. 與前幾週的銜接

- 第 2 週：`with open(...)`、例外處理、環境變數。
- 第 4 週：只依參考資料回答的 prompt。
- 第 6 週：Structured Outputs 與 JSON Schema。
- 第 7 週：`st.file_uploader()`、`.env`、`st.secrets`。
- 第 8 週：完成一個可操作的小型 LLM Web App。

本週聚焦「送進模型之前」的資料處理。第 10 週會把每個 chunk 轉成 embedding，
第 11 週再把檢索到的 chunks 組成 RAG context。


## 2. 文件處理的六層管線

| 層次 | 問題 | 常見錯誤 |
|---|---|---|
| 1. 輸入 | 使用者上傳了什麼？ | 沒有限制格式與大小 |
| 2. 格式路由 | 應交給哪個 reader？ | 把二進位 PDF 當文字 decode |
| 3. 文字抽取 | 是否真的取得文字？ | 掃描 PDF 沒有文字層 |
| 4. 清理 | 哪些空白可移除？ | 把段落全部黏在一起 |
| 5. Chunking | 每塊多大、重疊多少？ | overlap 大於 chunk size |
| 6. 應用 | 預覽、摘要或搜尋？ | 上傳時就自動呼叫付費 API |

<font color="#1565C0"><b>核心心智模型：</b></font>
Reader 解決「格式差異」，清理與 chunking 解決「下游應用需要一致輸入」。


In [ ]:
# Colab / 新環境第一次執行時取消下一行註解。
# %pip install pypdf python-docx pandas openai python-dotenv

from io import BytesIO
import csv
import json
import os
import re

import pandas as pd
from docx import Document
from pypdf import PdfReader

print("Week 9 文件處理環境載入完成")


## 3. 不同格式為何需要不同讀法？

- `.txt` / `.md`：通常可直接 decode，但仍可能遇到 UTF-8、Big5 等編碼差異。
- `.pdf`：檔案是二進位容器，要逐頁抽取文字；掃描影像 PDF 可能需要 OCR。
- `.docx`：Word 文件是壓縮封裝格式，要讀取段落與表格。
- `.csv`：列與欄本身有結構，轉成文字時應保留欄名與每列關係。

本週只處理「已有文字層」的 PDF。OCR、版面分析與表格辨識是進階延伸。


In [ ]:
def decode_text_bytes(data: bytes) -> str:
    """依常見編碼順序解碼純文字，全部失敗時提供明確錯誤。"""
    for encoding in ("utf-8-sig", "utf-8", "cp950"):
        try:
            return data.decode(encoding)
        except UnicodeDecodeError:
            continue
    raise ValueError("無法辨識文字編碼，請另存為 UTF-8 後再試。")


sample_bytes = "第一段：文件處理。\n第二段：準備 chunking。".encode("utf-8")
print(decode_text_bytes(sample_bytes))


In [ ]:
def extract_pdf_text(data: bytes) -> str:
    """逐頁抽取 PDF 文字，保留頁碼標記以利後續追蹤來源。"""
    reader = PdfReader(BytesIO(data))
    pages = []
    for page_number, page in enumerate(reader.pages, start=1):
        page_text = page.extract_text() or ""
        if page_text.strip():
            pages.append(f"[第 {page_number} 頁]\n{page_text.strip()}")
    if not pages:
        raise ValueError("PDF 沒有可抽取文字；它可能是掃描檔，需要 OCR。")
    return "\n\n".join(pages)


### PDF 常見限制

`pypdf` 擷取的是 PDF 的文字層，不保證還原視覺閱讀順序。雙欄排版、複雜表格、
頁首頁尾與掃描影像都可能造成結果不理想。正式 App 應讓使用者先預覽抽取結果，
不要直接假設「讀得到 PDF」就等於「文字完全正確」。


In [ ]:
def extract_docx_text(data: bytes) -> str:
    """讀取 Word 段落與表格，將內容轉成統一文字。"""
    document = Document(BytesIO(data))
    blocks = [p.text.strip() for p in document.paragraphs if p.text.strip()]

    for table_index, table in enumerate(document.tables, start=1):
        blocks.append(f"[表格 {table_index}]")
        for row in table.rows:
            values = [cell.text.strip() for cell in row.cells]
            blocks.append(" | ".join(values))

    if not blocks:
        raise ValueError("Word 文件沒有可讀取的段落或表格文字。")
    return "\n".join(blocks)


In [ ]:
def extract_csv_text(data: bytes, max_rows: int = 200) -> str:
    """保留欄名並將 CSV 前 max_rows 列轉成適合摘要的文字。"""
    text = decode_text_bytes(data)
    frame = pd.read_csv(BytesIO(text.encode("utf-8")))
    if frame.empty:
        raise ValueError("CSV 沒有資料列。")

    limited = frame.head(max_rows).fillna("")
    lines = [f"欄位：{', '.join(map(str, limited.columns))}"]
    for index, row in limited.iterrows():
        fields = [f"{column}={row[column]}" for column in limited.columns]
        lines.append(f"第 {index + 1} 列：" + "；".join(fields))
    return "\n".join(lines)


### CSV 不一定要全部轉成文字

若目標是計算平均、篩選或繪圖，應優先用 pandas 做確定性的資料處理；
若目標是請模型整理欄位意義、觀察趨勢或產生敘述，才把必要範圍轉成文字。
本例限制列數，是為了避免一次送入過量資料與不必要成本。


In [ ]:
def extract_text(filename: str, data: bytes) -> str:
    """依副檔名路由到正確 reader，回傳統一的文字字串。"""
    suffix = filename.lower().rsplit(".", maxsplit=1)[-1]
    readers = {
        "txt": decode_text_bytes,
        "md": decode_text_bytes,
        "pdf": extract_pdf_text,
        "docx": extract_docx_text,
        "csv": extract_csv_text,
    }
    if suffix not in readers:
        raise ValueError(f"不支援 .{suffix}；請使用 PDF、DOCX、CSV、TXT 或 MD。")
    return readers[suffix](data)


print(extract_text("sample.md", b"# Week 9\nDocument pipeline"))


## 4. 文字清理：不是刪得越乾淨越好

清理目標是降低噪音，但保留語意邊界：

- 同一行連續空白可合併。
- 三個以上空行可縮成兩個換行。
- 段落換行、頁碼標記與表格列通常應保留。
- 不應任意刪除標點、數字、日期與欄名。

清理前後都要能預覽；若清理規則破壞內容，下游摘要與搜尋也會一起變差。


In [ ]:
def clean_text(text: str) -> str:
    """統一換行、移除行內多餘空白，並保留段落分隔。"""
    normalized = text.replace("\r\n", "\n").replace("\r", "\n")
    lines = [re.sub(r"[ \t]+", " ", line).strip() for line in normalized.split("\n")]
    cleaned = "\n".join(lines)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)
    return cleaned.strip()


messy = "標題   有多餘空白\r\n\r\n\r\n第二段\t內容"
print(clean_text(messy))


## 5. Chunking 基本概念

長文件通常不能當成單一資料單位。Chunking 是把文字切成多個可管理片段：

- `chunk_size`：每塊最多保留多少字元。
- `overlap`：相鄰兩塊重複多少字元，降低句子剛好被切斷的風險。
- metadata：記錄來源檔名、chunk 編號、頁碼等資訊。

| 設定 | 可能結果 |
|---|---|
| chunk 太小 | 語意不完整、搜尋結果缺上下文 |
| chunk 太大 | 搜尋不精準、輸入成本提高 |
| overlap 太少 | 邊界資訊容易遺失 |
| overlap 太多 | 重複資料與成本增加 |

本週先用「字元長度」理解流程；第 10 週再討論 embedding 與語意搜尋效果。


In [ ]:
def chunk_text(text: str, chunk_size: int = 800, overlap: int = 120) -> list[dict]:
    """用滑動視窗切分文字，回傳含 chunk_id 與字元位置的資料。"""
    if chunk_size <= 0:
        raise ValueError("chunk_size 必須大於 0。")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap 必須介於 0（含）與 chunk_size（不含）之間。")

    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        content = text[start:end].strip()
        if content:
            chunks.append({
                "chunk_id": len(chunks),
                "start": start,
                "end": end,
                "text": content,
            })
        if end == len(text):
            break
        start = end - overlap
    return chunks


In [ ]:
demo_text = clean_text(
    "第一段介紹文件讀取。" * 12
    + "\n\n"
    + "第二段介紹文字清理。" * 12
    + "\n\n"
    + "第三段介紹 chunking。" * 12
)
demo_chunks = chunk_text(demo_text, chunk_size=120, overlap=20)

for chunk in demo_chunks:
    print(
        f"chunk={chunk['chunk_id']} "
        f"range={chunk['start']}:{chunk['end']} "
        f"length={len(chunk['text'])}"
    )
    print(chunk["text"][:60], "\n")


### 先觀察，再改良

固定字元切割容易從句子中間切開，但它的規則清楚、容易測試，適合第一次理解 chunking。
可延伸的做法包括：優先依段落切割、依句號切割、限制 token 數，或用語意邊界切割。
不論採哪種方法，都應保存來源 metadata，否則第 11 週很難顯示引用來源。


In [ ]:
def run_local_checks() -> None:
    """不呼叫 API，快速檢查清理、路由與 chunking 的關鍵邊界。"""
    assert clean_text("A   B\n\n\nC") == "A B\n\nC"
    assert extract_text("note.txt", "測試".encode("utf-8")) == "測試"
    assert len(chunk_text("A" * 250, chunk_size=100, overlap=20)) == 3

    invalid_cases = [
        {"chunk_size": 0, "overlap": 0},
        {"chunk_size": 100, "overlap": 100},
    ]
    for case in invalid_cases:
        try:
            chunk_text("測試", **case)
        except ValueError:
            pass
        else:
            raise AssertionError(f"應拒絕不合法設定：{case}")

    print("本機檢查通過：未呼叫付費 API")


run_local_checks()


## 6. AI 應該接在哪裡？

文件抽取、清理與 chunking 都是本機確定性流程，不必交給模型。
AI 適合接在管線後段，例如：

1. 對整份短文件產生摘要。
2. 對每個 chunk 提取關鍵字，再彙整。
3. 對 CSV 的小範圍資料產生敘述。

本週 App 只有使用者按下「產生 AI 摘要」才呼叫 Responses API。
上傳、預覽、清理與切塊都不產生 API 費用。


In [ ]:
from openai import OpenAI


def summarize_document(text: str, model: str | None = None) -> str:
    """用 Responses API 摘要已抽取文字；呼叫前先限制輸入長度。"""
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("找不到 OPENAI_API_KEY；請先設定環境變數或 Colab Secret。")

    client = OpenAI(api_key=api_key)
    selected_model = model or os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
    response = client.responses.create(
        model=selected_model,
        instructions=(
            "你是嚴謹的文件整理助理。只能根據文件內容回答；"
            "若文字擷取不完整，必須明確說明。"
        ),
        input=f"請用繁體中文整理摘要、5 個重點與資料限制：\n\n{text[:12000]}",
    )
    if not response.output_text:
        raise RuntimeError("模型沒有回傳文字結果。")
    return response.output_text


In [ ]:
# 付費 API 預設關閉。確認已設定測試用 API key 後再改成 True。
RUN_PAID_API = False

if RUN_PAID_API:
    print(summarize_document(demo_text))
else:
    print("已略過付費 API；本機文件處理功能仍可完整練習。")


## 7. 從 Notebook 到 Streamlit 專案

本週配套專案：`week09/week09_document_processor/`

| Notebook 概念 | 專案位置 |
|---|---|
| 格式路由與 reader | `document_utils.py` |
| 文字清理與 chunking | `document_utils.py` |
| OpenAI 摘要 helper | `app.py` |
| 上傳、參數與預覽 | `app.py` |
| 安裝與安全設定 | `requirements.txt`、`.env.example`、README |

執行：

```bash
cd week09/week09_document_processor
pip install -r requirements.txt
streamlit run app.py
```


## 8. 課堂練習 A：段落優先切割（必做）

改良固定字元切割：先用空行取得段落，再把過長段落切小。

驗收條件：

- 回傳格式仍為 `list[dict]`。
- 一般短段落不要從中間切開。
- 過長段落仍不能超過 `chunk_size`。
- 不可產生空 chunk。


In [ ]:
def chunk_by_paragraph(text: str, chunk_size: int = 800) -> list[dict]:
    """教師參考：優先合併完整段落，過長段落再用固定長度切割。"""
    if chunk_size <= 0:
        raise ValueError("chunk_size 必須大於 0。")

    paragraphs = [part.strip() for part in text.split("\n\n") if part.strip()]
    chunks = []
    buffer = ""

    def append_chunk(content: str) -> None:
        chunks.append({"chunk_id": len(chunks), "text": content})

    for paragraph in paragraphs:
        if len(paragraph) > chunk_size:
            if buffer:
                append_chunk(buffer)
                buffer = ""
            for start in range(0, len(paragraph), chunk_size):
                append_chunk(paragraph[start:start + chunk_size])
        elif not buffer:
            buffer = paragraph
        elif len(buffer) + 2 + len(paragraph) <= chunk_size:
            buffer += "\n\n" + paragraph
        else:
            append_chunk(buffer)
            buffer = paragraph

    if buffer:
        append_chunk(buffer)
    return chunks


paragraph_chunks = chunk_by_paragraph(demo_text, chunk_size=160)
print([(item["chunk_id"], len(item["text"])) for item in paragraph_chunks])


## 9. 課堂練習 B：文件品質報告（必做）

實作 `build_document_report()`，回傳：

- 檔名
- 清理後字元數
- chunk 數
- 平均 chunk 長度
- 是否可能需要 OCR

這份報告是第 10 週建立 embedding 前的資料品質檢查。


In [ ]:
def build_document_report(filename: str, cleaned_text: str, chunks: list[dict]) -> dict:
    """教師參考：建立可記錄、可測試的文件處理摘要。"""
    lengths = [len(item["text"]) for item in chunks]
    return {
        "filename": filename,
        "character_count": len(cleaned_text),
        "chunk_count": len(chunks),
        "average_chunk_length": round(sum(lengths) / len(lengths), 1) if lengths else 0,
        "possible_ocr_needed": filename.lower().endswith(".pdf") and not cleaned_text.strip(),
    }


report = build_document_report("demo.txt", demo_text, demo_chunks)
print(json.dumps(report, ensure_ascii=False, indent=2))


## 10. 課堂練習 C：Streamlit 功能改造（挑戰）

從下列方向選一項：

1. 新增 chunk 下載為 JSON。
2. 新增「只摘要目前選取 chunk」。
3. 對 CSV 顯示欄位型別與缺值數。
4. 加入檔案大小限制與更明確錯誤訊息。

修改後需在 README 記錄功能、操作方式、限制與測試案例。


In [ ]:
challenge_plan = {
    "feature": "下載 chunks JSON",
    "input": "cleaned_text 與 chunking 參數",
    "output": "UTF-8 JSON 檔",
    "error_cases": ["沒有上傳檔案", "chunks 為空"],
    "manual_test": "下載後重新用 json.load() 解析",
}
print(json.dumps(challenge_plan, ensure_ascii=False, indent=2))


## 11. 完成檢核

- [ ] 能說明 PDF、DOCX、CSV 與純文字為何不能共用同一 reader。
- [ ] 能預覽抽取文字並辨識掃描 PDF 限制。
- [ ] 能解釋清理規則保留了哪些語意邊界。
- [ ] 能調整 `chunk_size` 與 `overlap`，並拒絕不合法參數。
- [ ] 本機檢查不需 API key 即可通過。
- [ ] API key 未寫進程式碼、Notebook 或 Git。
- [ ] 知道上傳與預覽不應自動觸發付費 API。
- [ ] 完成至少一項課堂練習。


## 12. 常見問題

**PDF 顯示空白？**  
先確認是否為掃描影像。`pypdf` 不等於 OCR。

**中文 TXT 亂碼？**  
優先另存 UTF-8；本教材也示範 UTF-8-SIG 與 CP950 fallback。

**CSV 讀取失敗？**  
檢查編碼、分隔符號、欄名與壞掉的列。真實系統要允許使用者選 delimiter。

**Chunk 數量異常多？**  
檢查 `chunk_size` 是否太小、`overlap` 是否接近 `chunk_size`。

**為什麼不直接把整份文件送給模型？**  
長度、成本、等待時間與檢索精度都需要管理；而且第 10、11 週需要 chunk 作為資料單位。


## 13. 課後任務

使用一份不含機密的自備 PDF、DOCX 或 CSV：

1. 截圖原始文件、抽取預覽與 chunk 預覽。
2. 記錄檔案格式、字元數、chunk 數與參數。
3. 說明至少一個抽取或清理限制。
4. 若執行 AI 摘要，核對摘要是否只根據文件內容。
5. 將修改後專案與 README 推送到自己的 GitHub repo。


## 14. 下週預告：Embedding 與語意搜尋

本週的輸出不是只有摘要，而是一組含來源資訊的 chunks。
第 10 週會把每個 chunk 轉成向量，計算查詢與 chunks 的語意相似度，
再用 ChromaDB 或 FAISS 建立可搜尋的文件集合。

請保留本週專案與測試文件，下週會直接延伸。
